In [1]:
import mediapipe as mp
import cv2
import numpy as np

In [2]:
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from collections import deque


def calculate_angle(a, b, c):
    a = np.array(a)  # First
    b = np.array(b)  # Mid
    c = np.array(c)  # End
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
# Load the LSTM model


lstm_model = torch.jit.load(r'D:\PROGRAMMING\New folder\Rehabilitation-System\model_path\model_squats_scripted.pt')
lstm_model.eval()
# Initialize MediaPipe pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Video Capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables for counting and tracking
stage = ""
counter = 0

# Scaling angles
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(np.array([[0] * 7, [180] * 7]))

# Store pose sequences
timesteps = 50
pose_sequences = deque(maxlen=timesteps)
real_time_storage = []
frame_count = 0
collecting_real_time = False #this is a flag.false-during prediction, true-calculating real time values. 
predicted_vs_real_storage = [] 


with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert frame to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        # Detect pose
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Draw pose landmarks
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                      mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2),
                                      mp_drawing.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2))

            # Define angles to calculate
            angles_to_calculate = {
                "right_elbow_right_shoulder_right_hip": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                ],
                "left_elbow_left_shoulder_left_hip": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                ],
                "right_knee_mid_hip_left_knee": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                ],
                "right_hip_right_knee_right_ankle": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
                ],
                "left_hip_left_knee_left_ankle": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                ],
                "right_wrist_right_elbow_right_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                ],
                "left_wrist_left_elbow_left_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                ],
            }

            # Compute angles
            angles = [calculate_angle(*angles_to_calculate[key]) for key in angles_to_calculate]
            print(f"Calculated Angles: {angles}")

            # Input is normalized and added to sequence
            normalized_angles = scaler.transform([angles])
            pose_sequences.append(normalized_angles[0])
            
            # Collecting real-time frames(it checks if 20 new angles are appended,later forms paird of predictions and their actual angles)
            if collecting_real_time:#if flag is true
                real_time_storage.append(angles)
                if len(real_time_storage) == 20:
                    for pred, real in zip(predicted_angles, real_time_storage):#pairs predicted and real time values of same frame
                        predicted_vs_real_storage.append((pred, real)) 
                    collecting_real_time = False #reset to false so that next prediction cycle starts
                    real_time_storage = [] 
                    
            # Prediction
            if len(pose_sequences) == timesteps and not collecting_real_time:# if flag is false
                input_seq = torch.tensor([pose_sequences], dtype=torch.float32).to(device)
                with torch.no_grad():
                    predicted_normalized = lstm_model(input_seq).cpu().numpy().squeeze(0)
                predicted_angles = scaler.inverse_transform(predicted_normalized)
                
                print(f"Predicted Angles:{predicted_angles}")

                # Sliding window for input
                pose_sequences = deque(list(pose_sequences)[20:], maxlen=timesteps)#removes previous 20 frames
                pose_sequences.extend(real_time_storage)#appends new 20 frames to remaining 30 frames
                real_time_storage = []  
                collecting_real_time = True  
                
                # i have not focused on camera written display 
                for i, angle in enumerate(predicted_angles[0]):
                    cv2.putText(image, f'Predicted {i+1}: {int(angle)}', (50, 300 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # Show output
        cv2.imshow('Pose Estimation', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):

            break

cap.release()
cv2.destroyAllWindows()


Calculated Angles: [11.747638671277011, 42.39981497090211, 18.611776371550768, 177.80878328239606, 177.8150764018095, 177.05538665409284, 148.2550565339882]
Calculated Angles: [11.692742899398386, 41.30571239415751, 18.299867791739867, 177.99018132327913, 178.94141088354033, 123.08595914621202, 148.73854727616984]
Calculated Angles: [9.994759373636047, 40.10088959831018, 17.9802288318304, 177.98477794457165, 179.40452761405535, 157.84079457782897, 150.58272889066282]
Calculated Angles: [8.636941177408339, 39.384596725197305, 17.32667057934589, 177.76159164105042, 179.81206222018162, 166.35580549181785, 151.63668678420683]
Calculated Angles: [7.803761654980808, 38.634171611146726, 17.460278717535985, 176.97996410915624, 179.50731023303445, 165.57145980459356, 152.54158159702502]
Calculated Angles: [8.926544017279753, 39.58129667115577, 18.202534025562965, 176.68288026391437, 178.22667956895097, 9.219261654167916, 153.03908181825642]
Calculated Angles: [8.743728746170047, 38.822562812520

In [66]:
#this is for my check in output
print("Stored Predicted vs Real-Time Angle Pairs:")
for i, (pred, real) in enumerate(predicted_vs_real_storage):
    print(f"({pred}, {real})")

Stored Predicted vs Real-Time Angle Pairs:
([ 84.799706  65.24041   49.70664  149.4224   140.94109   70.97276
  89.59438 ], [85.5682560741304, 138.4200090375103, 30.123871175978028, 179.10051053800754, 179.29956659387727, 72.6900023735512, 92.73280387584862])
([ 80.76132  59.06883  62.10419 148.71906 143.87822  73.46203  83.4875 ], [86.60613144909142, 140.39239951921428, 44.48992257844003, 4.156399479322904, 2.403538799940937, 73.85938962819773, 89.1992263436797])
([ 78.26077  58.87952  64.32185 145.03302 141.35086  73.8394   85.7044 ], [90.66704223843033, 141.27050579318458, 174.6388650526659, 71.82776561809527, 129.57240932109838, 77.83369760262421, 87.87506913650708])
([ 78.14435  63.40362  67.66298 142.9032  137.80443  73.01916  84.70142], [89.20199160976973, 140.2784859211574, 30.960341720230623, 176.4684083667528, 176.46012157829347, 76.54653687658053, 86.21848060162012])
([ 70.05166  60.04724  68.22176 144.177   141.73901  61.53747  69.93577], [86.69380052027567, 138.84450796423

In [ ]:
#slicing and storing specific angles pred and actual vales
focus_angles=np.array([[pred[2:5],real[2:5]]for pred,real in predicted_vs_real_storage])
print(focus_angles)

[[[ 49.70663834 149.4223938  140.94108582]
  [ 30.12387118 179.10051054 179.29956659]]

 [[ 62.10419083 148.71905518 143.8782196 ]
  [ 44.48992258   4.15639948   2.4035388 ]]

 [[ 64.32185364 145.03302002 141.3508606 ]
  [174.63886505  71.82776562 129.57240932]]

 [[ 67.66297913 142.90319824 137.8044281 ]
  [ 30.96034172 176.46840837 176.46012158]]

 [[ 68.22176361 144.17700195 141.73901367]
  [ 22.07235567 179.40344184 175.56484587]]

 [[ 63.872612   146.89680481 144.29692078]
  [ 20.48152317 177.69506824 175.09408285]]

 [[ 59.5259552  143.62374878 142.02964783]
  [ 20.40886567 178.21003714 176.00034209]]

 [[ 60.61809158 142.63035583 139.83184814]
  [ 20.19932776 178.32111393 176.28411663]]

 [[ 61.7211647  140.13204956 139.63540649]
  [ 19.69673884 178.43023864 176.4555903 ]]

 [[ 61.66527176 134.02418518 134.00881958]
  [ 19.31425988 178.48826425 176.71036393]]

 [[ 63.0981369  129.90792847 131.49671936]
  [ 19.02779706 178.47549926 176.91610139]]

 [[ 65.51656342 119.60076141 120

In [ ]:
# #!/usr/bin/env python
 
# import cv2
# import time
 
# if __name__ == '__main__' :
 
#     # Start default camera
#     video = cv2.VideoCapture(0);
 
#     # Find OpenCV version
#     (major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
 
#     # With webcam get(CV_CAP_PROP_FPS) does not work.
#     # Let's see for ourselves.
 
#     if int(major_ver)  < 3 :
#         fps = video.get(cv2.cv.CV_CAP_PROP_FPS)
#         print("Frames per second using video.get(cv2.cv.CV_CAP_PROP_FPS): {0}".format(fps))
#     else :
#         fps = video.get(cv2.CAP_PROP_FPS)
#         print("Frames per second using video.get(cv2.CAP_PROP_FPS) : {0}".format(fps))
 
#     # Number of frames to capture
#     num_frames = 50;
 
#     print("Capturing {0} frames".format(num_frames))
 
#     # Start time
#     start = time.time()
 
#     # Grab a few frames
#     for i in range(0, num_frames) :
#         ret, frame = video.read()
 
#     # End time
#     end = time.time()
 
#     # Time elapsed
#     seconds = end - start
#     print ("Time taken : {0} seconds".format(seconds))
 
#     # Calculate frames per second
#     fps  = num_frames / seconds
#     print("Estimated frames per second : {0}".format(fps))
 
#     # Release video
#     video.release()

Frames per second using video.get(cv2.CAP_PROP_FPS) : 30.0
Capturing 50 frames
Time taken : 1.8902583122253418 seconds
Estimated frames per second : 26.451411257722004


# Make Basic Detection

In [3]:
# Getting video feed
cap = cv2.VideoCapture(0)
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False 

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True 
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass

        # Rendering
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2),
                                  )

        cv2.imshow('Raw Webcam Feed', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# 2. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

In [4]:
landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]

NameError: name 'landmarks' is not defined

In [ ]:
landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]

x: 1.01653159
y: 0.86054343
z: -1.10317266
visibility: 0.545346856

# 3. Calculate Angles

In [5]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

In [6]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]


NameError: name 'landmarks' is not defined

In [7]:
shoulder, elbow, wrist

NameError: name 'shoulder' is not defined

In [ ]:
calculate_angle(shoulder, elbow, wrist)

158.37397025964307

# Holistic

In [ ]:
# mp_drawing = mp.solutions.drawing_utils
# mp_holistic = mp.solutions.holistic

In [ ]:
# import cv2
# import mediapipe as mp

# mp_drawing = mp.solutions.drawing_utils
# mp_drawing_styles = mp.solutions.drawing_styles
# mp_holistic = mp.solutions.holistic
# mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

# cap = cv2.VideoCapture(0)

# # Initiate holistic model
# with mp_holistic.Holistic(min_detection_confidence=0.5, 
#                           min_tracking_confidence=0.5,
#                           static_image_mode=False,
#                           smooth_landmarks=True,
#                           model_complexity=2
#                             ) as holistic:
    
#     while cap.isOpened():
#         ret, frame = cap.read()
        
#         # Recolor Feed
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False        
        
#         # Make Detections
#         results = holistic.process(image)
        
#         # Recolor image back to BGR for rendering
#         image.flags.writeable = True   
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
#         # 1. Draw face landmarks
#         mp_drawing.draw_landmarks(
#             image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
#             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
#             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
#         )
        
#         # 2. Right hand
#         mp_drawing.draw_landmarks(
#             image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
#         )

#         # 3. Left Hand
#         mp_drawing.draw_landmarks(
#             image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
#         )

#         # 4. Pose Detections
#         mp_drawing.draw_landmarks(
#             image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
#         )
                        
#         cv2.imshow('Holistic Webcam Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()


In [ ]:
# SPINE ANGLE

In [ ]:
# import cv2
# import mediapipe as mp
# import numpy as np

# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# def calculate_angle(a, b, c):
#     a = np.array(a)  # First point  
#     b = np.array(b)  # Midpoint
#     c = np.array(c)  # End point
    
#     radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
#     angle = np.abs(radians * 180.0 / np.pi)
    
#     if angle > 180.0:
#         angle = 360 - angle
        
#     return angle

# cap = cv2.VideoCapture(0)

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
        
#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make detection
#         results = pose.process(image)
        
#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
#         try:
#             landmarks = results.pose_landmarks.landmark

#             # Get coordinates for back straightness check
#             shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
#                         landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
#             hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
#                    landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
#             knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
#             # Calculate angle for back straightness
#             spine_angle = calculate_angle(shoulder, hip, knee)
            
#             # Visualize spine angle
#             cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
#                         (50, 50), 
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
#             # Get foot positions for feet check
#             left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
#             right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
#             # Check if feet are lifting off the ground
#             if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
#                 cv2.putText(image, "Feet not planted!", 
#                             (50, 100), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
#             else:
#                 cv2.putText(image, "Feet planted", 
#                             (50, 100), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
#         except:
#             pass
        
#         # Render detections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
#                                   mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
#         cv2.imshow('Squat Form Detection', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()


# Holistic

In [ ]:
# mp_drawing = mp.solutions.drawing_utils
# mp_holistic = mp.solutions.holistic

In [ ]:
# import cv2
# import mediapipe as mp

# mp_drawing = mp.solutions.drawing_utils
# mp_drawing_styles = mp.solutions.drawing_styles
# mp_holistic = mp.solutions.holistic
# mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

# cap = cv2.VideoCapture(0)

# # Initiate holistic model
# with mp_holistic.Holistic(min_detection_confidence=0.5, 
#                           min_tracking_confidence=0.5,
#                           static_image_mode=False,
#                           smooth_landmarks=True,
#                           model_complexity=2
#                             ) as holistic:
    
#     while cap.isOpened():
#         ret, frame = cap.read()
        
#         # Recolor Feed
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False        
        
#         # Make Detections
#         results = holistic.process(image)
        
#         # Recolor image back to BGR for rendering
#         image.flags.writeable = True   
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
#         # 1. Draw face landmarks
#         mp_drawing.draw_landmarks(
#             image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
#             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
#             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
#         )
        
#         # 2. Right hand
#         mp_drawing.draw_landmarks(
#             image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
#         )

#         # 3. Left Hand
#         mp_drawing.draw_landmarks(
#             image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
#         )

#         # 4. Pose Detections
#         mp_drawing.draw_landmarks(
#             image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
#             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
#             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
#         )
                        
#         cv2.imshow('Holistic Webcam Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()


In [ ]:
# SPINE ANGLE

In [ ]:
# import cv2
# import mediapipe as mp
# import numpy as np

# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# def calculate_angle(a, b, c):
#     a = np.array(a)  # First point  
#     b = np.array(b)  # Midpoint
#     c = np.array(c)  # End point
    
#     radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
#     angle = np.abs(radians * 180.0 / np.pi)
    
#     if angle > 180.0:
#         angle = 360 - angle
        
#     return angle

# cap = cv2.VideoCapture(0)

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
        
#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make detection
#         results = pose.process(image)
        
#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
#         try:
#             landmarks = results.pose_landmarks.landmark

#             # Get coordinates for back straightness check
#             shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
#                         landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
#             hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
#                    landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
#             knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
#             # Calculate angle for back straightness
#             spine_angle = calculate_angle(shoulder, hip, knee)
            
#             # Visualize spine angle
#             cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
#                         (50, 50), 
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
#             # Get foot positions for feet check
#             left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
#             right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
#             # Check if feet are lifting off the ground
#             if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
#                 cv2.putText(image, "Feet not planted!", 
#                             (50, 100), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
#             else:
#                 cv2.putText(image, "Feet planted", 
#                             (50, 100), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
#         except:
#             pass
        
#         # Render detections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
#                                   mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
#         cv2.imshow('Squat Form Detection', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()
